# compilar_reduzir_classificadas — Google Colab

Compila os dados classificados: para cada imagem, exibe o **original (RGB)** ao lado do **predict**
(`_pred.tif`), e gera uma versão **reduzida** guardando só as regiões com fotovoltaica.

## Redução por patches
1. Fatia o original e o predict no **mesmo grid** de `PATCH_SIZE`.
2. Marca cada pedaço como *sobrevivente* se o **predict** tiver algum pixel = 1.
3. Descarta linhas/colunas do grid totalmente vazias.
4. **Remonta** os pedaços sobreviventes numa imagem compacta (original e predict alinhados).

> O geotransform da imagem reduzida é **aproximado** — remover as lacunas quebra a
> continuidade geoespacial. Serve para inspeção/dataset, não para medição de coordenadas.

---
**Drive montado em `/content/drive`.**

In [ ]:
# ── Célula 1: Montar Drive e instalar dependências ───────────────────────────
from google.colab import drive
drive.mount('/content/drive')

!pip install -q rasterio matplotlib tqdm

In [ ]:
# ── Célula 2: PARÂMETROS ─────────────────────────────────────────────────────
INPUT_DIR   = "/content/drive/MyDrive/DS_FV_TIFs_scaled/PREDICT_V2"           # originais (5 bandas)
OUTPUT_DIR  = "/content/drive/MyDrive/DL_fotovoltaica/tif_classificadas_2025"  # predicts *_pred.tif
REDUCED_DIR = "/content/drive/MyDrive/DL_fotovoltaica/tif_reduzidas_2025"      # saída reduzida

PATCH_SIZE     = 256      # tamanho do pedaço (px) do grid de redução
RGB_BANDS      = (3, 2, 1) # bandas 1-based p/ composição RGB
THRESHOLD_PRED = 0.5      # usado só se o predict for float (senão, ==1)
MAX_VIS        = 1       # nº máximo de pares a visualizar na célula de visualização

print("Parâmetros carregados.")
print(f"  INPUT_DIR  : {INPUT_DIR}")
print(f"  OUTPUT_DIR : {OUTPUT_DIR}")
print(f"  REDUCED_DIR: {REDUCED_DIR}")
print(f"  PATCH_SIZE={PATCH_SIZE}px | RGB_BANDS={RGB_BANDS}")

In [ ]:
# ── Célula 3: Imports ────────────────────────────────────────────────────────
from pathlib import Path
import numpy as np
import rasterio
import rasterio.transform
import matplotlib.pyplot as plt

try:
    from tqdm import tqdm
except ImportError:
    def tqdm(it, **kw):
        return it

input_dir   = Path(INPUT_DIR)
output_dir  = Path(OUTPUT_DIR)
reduced_dir = Path(REDUCED_DIR)
reduced_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Célula 4: Parear original ↔ predict ──────────────────────────────────────
def parear(input_dir: Path, output_dir: Path):
    pares = []
    for pred in sorted(output_dir.glob('*_pred.tif')):
        stem = pred.stem[:-len('_pred')]
        orig = input_dir / f'{stem}.tif'
        if orig.exists():
            pares.append((orig, pred))
    return pares

pares = parear(input_dir, output_dir)
print(f'{len(pares)} par(es) original ↔ predict encontrados.')

In [ ]:
# ── Célula 5: Leitura e visualização lado a lado ─────────────────────────────
def ler_rgb(tif_path: Path, rgb_bands=RGB_BANDS) -> np.ndarray:
    with rasterio.open(tif_path) as src:
        bands = [src.read(b) for b in rgb_bands]
    rgb = np.stack(bands, axis=-1).astype(np.float32)
    for i in range(rgb.shape[-1]):
        ch = rgb[..., i]
        lo, hi = np.percentile(ch, (2, 98))
        rgb[..., i] = np.clip((ch - lo) / (hi - lo + 1e-6), 0, 1)
    return rgb

def ler_mascara(pred_path: Path) -> np.ndarray:
    with rasterio.open(pred_path) as src:
        data = src.read(1)
    return (data == 1) if data.dtype == np.uint8 else (data > THRESHOLD_PRED)

def visualizar_par(orig_path: Path, pred_path: Path):
    rgb  = ler_rgb(orig_path)
    mask = ler_mascara(pred_path)
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    ax[0].imshow(rgb);              ax[0].set_title(f'Original: {orig_path.name}'); ax[0].axis('off')
    ax[1].imshow(mask, cmap='gray');ax[1].set_title(f'Predict: {pred_path.name}');  ax[1].axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Célula 6: Redução por patches ────────────────────────────────────────────
def reduzir_por_patches(orig_path: Path, pred_path: Path,
                        patch_size: int = PATCH_SIZE,
                        threshold: float = THRESHOLD_PRED):
    """Fatia original+predict no mesmo grid de patch_size, mantém só os pedaços
    cujo predict tem algum pixel=1, descarta linhas/colunas totalmente vazias e
    remonta os sobreviventes em imagens compactas alinhadas.

    Retorna (orig_red (C,Hr,Wr), pred_red (Hr,Wr), meta) — ou (None, None, None)
    se a imagem não tiver nenhum pixel positivo."""
    with rasterio.open(orig_path) as src:
        orig = src.read()                 # (C, H, W)
        transform, crs = src.transform, src.crs
    with rasterio.open(pred_path) as src:
        pred = src.read(1)                # (H, W)

    C, H, W = orig.shape
    Hc, Wc = min(H, pred.shape[0]), min(W, pred.shape[1])   # alinha dimensões
    orig, pred = orig[:, :Hc, :Wc], pred[:Hc, :Wc]
    mask = (pred == 1) if pred.dtype == np.uint8 else (pred > threshold)

    n_rows = int(np.ceil(Hc / patch_size))
    n_cols = int(np.ceil(Wc / patch_size))

    # grid booleano: célula sobrevive se o predict tiver algum pixel=1 nela
    surv = np.zeros((n_rows, n_cols), dtype=bool)
    for gr in range(n_rows):
        for gc in range(n_cols):
            r0, c0 = gr * patch_size, gc * patch_size
            r1, c1 = min(r0 + patch_size, Hc), min(c0 + patch_size, Wc)
            if mask[r0:r1, c0:c1].any():
                surv[gr, gc] = True

    if not surv.any():
        return None, None, None

    keep_rows = np.where(surv.any(axis=1))[0]
    keep_cols = np.where(surv.any(axis=0))[0]

    def cell_h(gr): return min((gr + 1) * patch_size, Hc) - gr * patch_size
    def cell_w(gc): return min((gc + 1) * patch_size, Wc) - gc * patch_size
    row_h = [cell_h(int(gr)) for gr in keep_rows]
    col_w = [cell_w(int(gc)) for gc in keep_cols]
    Hr, Wr = sum(row_h), sum(col_w)

    orig_red = np.zeros((C, Hr, Wr), dtype=orig.dtype)
    pred_red = np.zeros((Hr, Wr), dtype=pred.dtype)

    roff = 0
    for i, gr in enumerate(keep_rows):
        coff = 0
        for j, gc in enumerate(keep_cols):
            if surv[gr, gc]:              # célula vazia dentro de linha/col mantida = zero-fill
                r0, c0 = int(gr) * patch_size, int(gc) * patch_size
                orig_red[:, roff:roff + row_h[i], coff:coff + col_w[j]] = orig[:, r0:r0 + row_h[i], c0:c0 + col_w[j]]
                pred_red[roff:roff + row_h[i], coff:coff + col_w[j]]    = pred[r0:r0 + row_h[i], c0:c0 + col_w[j]]
            coff += col_w[j]
        roff += row_h[i]

    # geotransform APROXIMADO: origem = canto sup-esq do 1º patch mantido
    r_top  = int(keep_rows[0]) * patch_size
    c_left = int(keep_cols[0]) * patch_size
    west   = transform.c + c_left * transform.a
    north  = transform.f + r_top  * transform.e
    new_transform = rasterio.transform.from_origin(west, north, transform.a, -transform.e)

    meta = {'crs': crs, 'transform': new_transform,
            'orig_shape': (C, Hc, Wc), 'red_shape': (C, Hr, Wr),
            'patches_total': int(surv.size), 'patches_mantidos': int(surv.sum())}
    return orig_red, pred_red, meta

In [ ]:
# ── Célula 7: Salvar imagem reduzida ─────────────────────────────────────────
def salvar_reduzido(stem: str, orig_red, pred_red, meta, reduced_dir: Path):
    C, Hr, Wr = orig_red.shape
    orig_out = reduced_dir / f'{stem}_img_reduzido.tif'
    pred_out = reduced_dir / f'{stem}_pred_reduzido.tif'

    with rasterio.open(orig_out, 'w', driver='GTiff', height=Hr, width=Wr, count=C,
                       dtype=orig_red.dtype, crs=meta['crs'], transform=meta['transform'],
                       compress='lzw') as dst:
        dst.write(orig_red)

    with rasterio.open(pred_out, 'w', driver='GTiff', height=Hr, width=Wr, count=1,
                       dtype=pred_red.dtype, crs=meta['crs'], transform=meta['transform'],
                       compress='lzw') as dst:
        dst.write(pred_red, 1)

    return orig_out, pred_out

In [ ]:
# ── Célula 8: Visualizar alguns pares (original | predict) ───────────────────
for orig, pred in pares[:MAX_VIS]:
    visualizar_par(orig, pred)

In [ ]:
# ── Célula 9: Loop principal — reduzir e salvar todos ────────────────────────
n_red = n_semfv = n_erro = 0
for orig, pred in tqdm(pares, desc='reduzindo'):
    stem = pred.stem[:-len('_pred')]
    try:
        orig_red, pred_red, meta = reduzir_por_patches(orig, pred)
        if orig_red is None:
            n_semfv += 1
            continue
        salvar_reduzido(stem, orig_red, pred_red, meta, reduced_dir)
        n_red += 1
    except Exception as e:
        n_erro += 1
        print('erro em', stem, ':', e)

print(f'\nReduzidas: {n_red} | sem FV: {n_semfv} | erros: {n_erro} | total: {len(pares)}')
print('Saída em:', reduced_dir)

In [ ]:
# ── Célula 10: Conferir uma imagem reduzida (original reduzido | predict reduzido) ──
reduzidos = sorted(reduced_dir.glob('*_img_reduzido.tif'))
if reduzidos:
    ex   = reduzidos[0]
    stem = ex.stem[:-len('_img_reduzido')]
    print('Exemplo:', stem)
    visualizar_par(ex, reduced_dir / f'{stem}_pred_reduzido.tif')
else:
    print('Nenhuma imagem reduzida encontrada em', reduced_dir)